In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "numpy", "matplotlib"]
ASSETS = [{"name": "credit_data.csv", "url": "/labs/1.1.3/credit_data.csv", "size": 582250, "sha256": "5cd12574c8bc729014e3212feee325fd65ddd22c875a2561f013e7f78eeba8d7"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 1.1.3 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# 读取数据集
data = pd.read_csv('credit_data.csv')

In [ ]:
# 1. 数据完整性审核
missing_values = data._________       #数据缺失值统计 2分
duplicate_values = data._________     #数据重复值统计 2分
# 输出结果
print("缺失值统计:")
print(missing_values)
print("重复值统计:")
print(duplicate_values)

In [ ]:
# 2. 数据合理性审核
data['is_age_valid'] = _________._________(18, 70)              #Age数据的合理性审核 2分
data['is_income_valid'] = _________ > _________                 #Income数据的合理性审核 2分
data['is_loan_amount_valid'] = _________ < (_________ * 5)      #LoanAmount数据的合理性审核 2分
data['is_credit_score_valid'] = _________._________(300, 850)   #CreditScore数据的合理性审核 2分
# 合理性检查结果
validity_checks = data[['is_age_valid', 'is_income_valid', 'is_loan_amount_valid', 'is_credit_score_valid']].all(axis=1)
data['is_valid'] = validity_checks
# 输出结果
print("数据合理性检查:")
print(data[['is_age_valid', 'is_income_valid', 'is_loan_amount_valid', 'is_credit_score_valid', 'is_valid']].describe())

In [ ]:
# 3. 数据清洗和异常值处理
# 标记不合理数据
invalid_rows = data[~data['is_valid']]
# 删除不合理数据行
cleaned_data = data[data['is_valid']]
# 删除标记列
cleaned_data = cleaned_data.drop(columns=['is_age_valid', 'is_income_valid', 'is_loan_amount_valid', 'is_credit_score_valid', 'is_valid'])
# 保存清洗后的数据
_________._________(_________, index=False)
print("数据清洗完成，已保存为 'cleaned_credit_data.csv'")